# UTS Graf Sains Data: Tugas Pendahuluan

# Soal No 1.1 Persiapan dan Perkenalan Lingkungan

| Langkah | Linux / macOS (soal) | Windows PowerShell / terminal VS Code |
|---|---|---|
| Buat environment | `python -m venv myenv` | `python -m venv myenv` |
| Nonaktifkan environment lain | `conda deactivate` | `conda deactivate` (jika memakai conda) |
| Aktifkan environment | `source myenv/bin/activate` | `myenv\Scripts\Activate.ps1` |
| Install dependensi | `pip install -r requirements.txt` | `pip install -r requirements.txt` |


In [19]:
import sys, importlib.metadata as md_
print("Python        :", sys.version.split()[0])
print("Interpreter   :", sys.executable)
for paket in ["neo4j", "pandas", "tqdm"]:
    try:
        print(f"{paket:<14}:", md_.version(paket))
    except md_.PackageNotFoundError:
        print(f"{paket:<14}: BELUM TERPASANG (jalankan: pip install {paket})")

Python        : 3.14.3
Interpreter   : c:\Users\HP\AppData\Local\Programs\Python\Python314\python.exe
neo4j         : 6.4.1
pandas        : 2.3.3
tqdm          : 4.70.1


In [20]:
pip install neo4j

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Soal No 1.2 Mempelajari Penggunaan Driver Neo4J

## 1. Ikuti tutorial penggunaan driver Neo4J untuk melakukan pembuatan objek dan akses objek berikut.

## 2. Baris berikut mendefinisikan string koneksi ke database Neo4J

In [ ]:
import os
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"

if os.path.exists("password.txt"):
    PASSWORD = open("password.txt", encoding="utf-8").read().strip()
else:
    PASSWORD = os.getenv("NEO4J_PASSWORD", "")
assert PASSWORD, "Password Neo4j tidak ditemukan. Buat file password.txt atau isi variabel lingkungan NEO4J_PASSWORD."

AUTH = ("neo4j", PASSWORD)

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Terhubung")

Terhubung


## 3. Baris perintah berikut melakukan pembuatan sebuah Node berikut dengan relationship-nya.

In [23]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_="neo4j",
    )

## 4. Untuk mencetak dan menjalankan keseluruhan

In [24]:
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


# Pertanyaan Pendahuluan (Checkpint)

## 1. Apa nama Database yang digunakan pada tutorial ini?

Nama database: neo4j (database default), terlihat dari database_="neo4j".

## 2. Sebutkan akses lain (seperti sintaks DDL/DML CYPHER) yang dapat digunakan mengakses database lain dalam 1 instace?

Akses ke database lain dalam 1 instance:
- *Python:* : database_="namadb" pada execute_query, atau driver.session(database="namadb").
- *Cypher:* : use namadb (Neo4j Browser) atau awalan USE namadb pada query.
- *DDL administrasi (dijalankan di database system):* : CREATE DATABASE namadb, SHOW DATABASES, DROP DATABASE namadb.

## 3. Apa jenis graf pertemanan yang telah dibuat disini?

- graf berarah (directed) karena (a)-[:KNOWS]->(friend). 
- Grafnya juga homogen (satu label simpul, Person) dan tanpa bobot. 
- Arti arah: Arthur mengenal Guinevere tidak otomatis berarti sebaliknya.

# CYPHER total Reset

Jalankan perintah berikut untuk melakukan truncate pada konten database.

In [25]:
# Hati-hati: menghapus SEMUA isi database
with GraphDatabase.driver(URI, auth=AUTH) as drv:
    drv.execute_query("MATCH (n) DETACH DELETE n", database_="neo4j")
print("Database dikosongkan")

Database dikosongkan


# Soal No 1.3 Melakukan proses Impor Data dari RetailRetailRocket ke Database Neo4J

## 1. Unduh data dari kaggle dengan perintah berikut

Sumber data : Roman Zykov

In [26]:
import os, zipfile, argparse
import pandas as pd
import math
from tqdm import tqdm
from neo4j import GraphDatabase

ZIP = r"D:\baru 1\archive.zip"
SRC = r"D:\baru 1\data"
DB  = "neo4j"

os.makedirs(SRC, exist_ok=True)
if not os.path.exists(os.path.join(SRC, "events.csv")):
    with zipfile.ZipFile(ZIP) as z:
        print(z.namelist())
        z.extractall(SRC)
print(os.listdir(SRC))

['category_tree.csv', 'events.csv', 'item_properties_part1.csv', 'item_properties_part2.csv']
['category_tree.csv', 'events.csv', 'item_properties_part1.csv', 'item_properties_part2.csv']


## 2. Perhatikan fitur yang telah diunduh dengan rincian sebagai berikut:
(a). timestamp : Waktu dalam format unix epoch

(b). visitorid : ID unik pengunjung

(c). event : Jenis transaksi default : view, addtocart, transaction

(d). itemid : ID unik produk

(e). transactionid : ID transaksi, hanya terisi saat event berjenis transaction

In [27]:
df = pd.read_csv(os.path.join(SRC, "events.csv"))
df["dt"] = pd.to_datetime(df["timestamp"], unit="ms")

print(df.shape)
print(df.dtypes)
print(df.head())
print("Visitor unik :", df.visitorid.nunique())
print("Item unik    :", df.itemid.nunique())
print(df.event.value_counts())
print("transactionid terisi:", df.transactionid.notna().sum())
print("Rentang waktu:", df.dt.min(), "s.d.", df.dt.max())

(2756101, 6)
timestamp                 int64
visitorid                 int64
event                    object
itemid                    int64
transactionid           float64
dt               datetime64[ns]
dtype: object
       timestamp  visitorid event  itemid  transactionid  \
0  1433221332117     257597  view  355908            NaN   
1  1433224214164     992329  view  248676            NaN   
2  1433221999827     111016  view  318965            NaN   
3  1433221955914     483717  view  253185            NaN   
4  1433221337106     951259  view  367447            NaN   

                       dt  
0 2015-06-02 05:02:12.117  
1 2015-06-02 05:50:14.164  
2 2015-06-02 05:13:19.827  
3 2015-06-02 05:12:35.914  
4 2015-06-02 05:02:17.106  
Visitor unik : 1407580
Item unik    : 235061
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64
transactionid terisi: 22457
Rentang waktu: 2015-05-03 03:00:04.384000 s.d. 2015-09-18 02:59:47.788000


## 3. Buatlah mekanisme pembuatan session Neo4J baru dengan perintah kurang lebih sepeti berikut
(sumber: Neo4J Driver 6.4)

In [28]:
def parse_arguments(argv=None):
    parser = argparse.ArgumentParser(
        description="Impor dataset RetailRocket (events.csv) ke database Neo4J."
    )
    parser.add_argument("-u", "--user", default="neo4j", help="Username Neo4j (default: neo4j)")
    parser.add_argument("-p", "--password", default="neo4j", help="Password Neo4J (default: neo4j)")
    parser.add_argument("-b", "--bolt", default="neo4j://localhost:7687",
                        help="Neo4J URI / Bolt (default: neo4j://localhost:7687)")
    parser.add_argument("-s", "--source", default="data", help="Direktori/folder sumber data (default: data)")
    return parser.parse_args(argv)

# Di notebook argumen diisi manual. URI & password diambil dari sel koneksi di atas (URI, AUTH),
args = parse_arguments(["-u", AUTH[0], "-p", AUTH[1], "-b", URI, "-s", SRC])

# Cek keberadaan events.csv
csv_file_path = os.path.join(args.source, "events.csv")
if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"Error: File '{csv_file_path}' tidak ditemukan!")

# Koneksi (tes langsung ke database DB)
try:
    driver = GraphDatabase.driver(args.bolt, auth=(args.user, args.password))
    driver.execute_query("RETURN 1 AS ok", database_=DB)
    print("Terhubung ke database:", DB)
except Exception as e:
    raise RuntimeError(f"Gagal terhubung ke database: {e}")

# Mekanisme session baru
session = driver.session(database=DB)
session.close()

Terhubung ke database: neo4j


## 5. Baris perintah untuk mengimpor data dalam CYPHER, contohnya:

*Sel-sel no. 5 dijalankan lebih dulu, karena fungsi `create_constraints` dan `import_batch` dipanggil oleh proses impor di no. 4.*

In [29]:
def create_constraints(session):
    session.run("CREATE CONSTRAINT visitor_id IF NOT EXISTS FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE")
    session.run("CREATE CONSTRAINT item_id IF NOT EXISTS FOR (i:Item) REQUIRE i.itemId IS UNIQUE")

# Versi yang dipakai: FOREACH + CASE (hasil sama dengan CALL/UNION ALL, dan lebih aman di berbagai versi Neo4j)
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})
    FOREACH (ignore IN CASE WHEN row.event = 'view' THEN [1] ELSE [] END |
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i))
    FOREACH (ignore IN CASE WHEN row.event = 'addtocart' THEN [1] ELSE [] END |
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i))
    FOREACH (ignore IN CASE WHEN row.event = 'transaction' THEN [1] ELSE [] END |
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i))
    """
    tx.run(cypher_query, batch=batch)

In [30]:
# Versi sesuai contoh soal (CALL + UNION ALL), diberi nama berbeda agar tidak menimpa import_batch di atas.
# Untuk memakainya: ganti session.execute_write(import_batch, ...) menjadi session.execute_write(import_batch_call, ...)
def import_batch_call(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})

    WITH v, i, row
    CALL {
      WITH v, i, row
      WITH v, i, row WHERE row.event = 'view'
      CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

      UNION ALL

      WITH v, i, row
      WITH v, i, row WHERE row.event = 'addtocart'
      CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

      UNION ALL

      WITH v, i, row
      WITH v, i, row WHERE row.event = 'transaction'
      CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

## 4. Lanjutkan dengan proses impor dengan mekanisme batch, kurang lebih sebagai berikut:

In [31]:
MAX_ROWS = 50_000     
batch_size = 5000

with driver.session(database=DB) as session:
    session.run("MATCH (n) DETACH DELETE n")   
    create_constraints(session)

    print(f"Membaca file data dari: {csv_file_path}")
    chunks = pd.read_csv(csv_file_path, chunksize=batch_size, nrows=MAX_ROWS)

    if MAX_ROWS:
        total_rows = MAX_ROWS
    else:
        with open(csv_file_path, encoding="utf-8") as f:
            total_rows = sum(1 for _ in f) - 1
    total_chunks = math.ceil(total_rows / batch_size)

    print("Memulai proses impor data ke Neo4j...")
    for chunk in tqdm(chunks, total=total_chunks, desc="Mengimpor Batch Data"):
        
        # Membersihkan data null/NaN pada transactionid
        chunk["transactionid"] = (
            chunk["transactionid"].fillna(-1).astype("int64").astype(str).replace("-1", "")
        )
        records = chunk.to_dict(orient="records")
        session.execute_write(import_batch, records)

print("Proses impor berhasil.")

Membaca file data dari: D:\baru 1\data\events.csv
Memulai proses impor data ke Neo4j...


Mengimpor Batch Data: 100%|██████████| 10/10 [00:06<00:00,  1.66it/s]

Proses impor berhasil.


In [32]:
for q in ["MATCH (n) RETURN labels(n)[0] AS jenis, count(n) AS jumlah",
          "MATCH ()-[r]->() RETURN type(r) AS relasi, count(r) AS jumlah"]:
    recs, _, _ = driver.execute_query(q, database_=DB)
    print([r.data() for r in recs])

[{'jenis': 'Item', 'jumlah': 24716}, {'jenis': 'Visitor', 'jumlah': 28860}]
[{'relasi': 'VIEWED', 'jumlah': 48415}, {'relasi': 'ADDED_TO_CART', 'jumlah': 1153}, {'relasi': 'BOUGHT', 'jumlah': 432}]
